[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/17_feature_engineering/03_interaction_datetime_text_aggregation.ipynb)

# Bài 03 — Tương tác, thời gian, văn bản và feature tổng hợp (aggregation)

**🎯 Mục tiêu**
- **Tương tác** giữa feature: khi nào model tuyến tính cần, khi nào cây tự học được.
- Feature **thời gian**: giờ, thứ, tháng, cuối tuần, ngày lễ — và **mã hoá chu kỳ** (sin/cos).
- Feature từ **văn bản** đơn giản: độ dài, đếm từ khoá, TF-IDF.
- Feature **tổng hợp theo nhóm** (aggregation): "giao dịch này lớn gấp bao nhiêu lần trung bình của khách hàng?" — và cách tính không rò rỉ.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/17_feature_engineering

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)

## 1. Tương tác

Hiệu quả khuyến mãi phụ thuộc **đồng thời** vào kênh và nhóm khách: giảm giá trên app chỉ tác động mạnh với khách trẻ. Model tuyến tính cộng tác động riêng lẻ — không biểu diễn được "phụ thuộc đồng thời" nếu không có feature tích $x_1 \cdot x_2$:

In [ ]:
n = 6000
young = rng.random(n) < 0.5
app = rng.random(n) < 0.5
discount = rng.uniform(0, 1, n)                                             # mức giảm giá (đã chuẩn hoá về 0–1)
logit = -1.5 + 6 * discount * young * app - 0.5 * young * (1 - app)         # giảm giá chỉ tác động khi trẻ VÀ dùng app
y_buy = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
X_base = np.c_[young, app, discount].astype(float)
X_inter = np.c_[X_base, young * app, young * discount, app * discount, young * app * discount]
cv = StratifiedKFold(5, shuffle=True, random_state=0)
for name, model, X_ in [("Logistic, feature gốc", LogisticRegression(max_iter=2000), X_base),
                        ("Logistic + feature tương tác", LogisticRegression(max_iter=2000), X_inter),
                        ("HistGB, feature gốc", HistGradientBoostingClassifier(random_state=0), X_base)]:
    print(f"{name:30s} ROC-AUC = {cross_val_score(model, X_, y_buy, cv=cv, scoring='roc_auc').mean():.4f}")

- Model tuyến tính **cần** feature tương tác do ta tạo (từ hiểu biết miền, hoặc `PolynomialFeatures(interaction_only=True)` — cẩn thận bùng nổ số feature).
- Cây **tự học** tương tác (split trên `young` rồi split trên `app` bên dưới).

## 2. Feature thời gian và mã hoá chu kỳ

Nhu cầu gọi xe theo giờ: cao lúc 8h và 18h; **23h gần 0h**. Mã hoá giờ là số 0–23 khiến model tuyến tính coi 23 và 0 cách xa nhau nhất. Mã hoá chu kỳ: $\left(\sin\frac{2\pi h}{24}, \cos\frac{2\pi h}{24}\right)$ — các giờ nằm trên một vòng tròn.

In [ ]:
hours = rng.integers(0, 24, 5000)
demand = 50 + 30 * np.exp(-((hours - 8) ** 2) / 4) + 40 * np.exp(-((hours - 18) ** 2) / 6) \
         + 15 * np.cos(2 * np.pi * hours / 24) + rng.normal(0, 5, 5000)
encodings = {
    "giờ là số": hours[:, None].astype(float),
    "sin/cos (1 harmonic)": np.c_[np.sin(2 * np.pi * hours / 24), np.cos(2 * np.pi * hours / 24)],
    "sin/cos (4 harmonic)": np.column_stack([f(2 * np.pi * k * hours / 24) for k in range(1, 5) for f in (np.sin, np.cos)]),
    "one-hot 24 giờ": OneHotEncoder(sparse_output=False).fit_transform(hours[:, None]),
}
grid_h = np.arange(24)
plt.scatter(hours + rng.uniform(-0.3, 0.3, 5000), demand, s=2, alpha=0.2, color="gray")
for name, Xh in encodings.items():
    r2 = cross_val_score(LinearRegression(), Xh, demand, cv=KFold(5, shuffle=True, random_state=0), scoring="r2").mean()
    m = LinearRegression().fit(Xh, demand)
    preds = [m.predict(Xh[hours == h][:1])[0] for h in grid_h]
    plt.plot(grid_h, preds, lw=2, label=f"{name}: R² = {r2:.3f}")
plt.xlabel("giờ")
plt.ylabel("nhu cầu")
plt.legend(fontsize=8)
plt.title("Linear Regression với các cách mã hoá giờ")
plt.show()

Các feature thời gian hữu ích khác: thứ trong tuần, cuối tuần, tháng/quý, ngày lễ (và **khoảng cách** tới ngày lễ gần nhất — mua sắm tăng trước Tết), ngày trả lương, "số ngày kể từ lần mua cuối" (recency).

In [ ]:
dates = pd.Series(pd.date_range("2024-01-01", periods=10, freq="37h"))
tet = pd.Timestamp("2024-02-10")
pd.DataFrame({"thời điểm": dates, "giờ": dates.dt.hour, "thứ": dates.dt.dayofweek, "cuối tuần": dates.dt.dayofweek >= 5,
              "tháng": dates.dt.month, "số ngày tới Tết": (tet - dates).dt.days, "giờ_sin": np.sin(2 * np.pi * dates.dt.hour / 24).round(3)})

## 3. Feature từ văn bản

Trước khi dùng model ngôn ngữ (chương 32), vài feature đơn giản thường rất mạnh: độ dài, số dấu "!", số từ khoá cảm xúc, tỉ lệ chữ hoa — và TF-IDF.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

reviews = pd.read_csv("../../data/sample/reviews_vi.csv")
simple = pd.DataFrame({"len": reviews.text.str.len(), "n_words": reviews.text.str.split().str.len(),
                       "n_exclaim": reviews.text.str.count("!"), "has_khong": reviews.text.str.lower().str.contains("không").astype(int)})
y_rev = reviews.label.to_numpy()
cv5 = StratifiedKFold(5, shuffle=True, random_state=0)
print(f"4 feature đơn giản        : ROC-AUC = {cross_val_score(LogisticRegression(max_iter=2000), StandardScaler().fit_transform(simple), y_rev, cv=cv5, scoring='roc_auc').mean():.3f}")
print(f"TF-IDF (unigram + bigram) : ROC-AUC = {cross_val_score(make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression(max_iter=2000)), reviews.text, y_rev, cv=cv5, scoring='roc_auc').mean():.3f}")

**TF-IDF** = tần suất từ trong văn bản × độ "hiếm" của từ trong toàn bộ tập: $\text{tfidf}(w, d) = \text{tf}(w, d)\cdot\log\frac{N}{\text{df}(w)}$ — từ xuất hiện ở mọi văn bản ("sản phẩm", "mua") có trọng số thấp. Chi tiết ở chương 32.

## 4. Feature tổng hợp theo nhóm (aggregation)

Phát hiện gian lận: một giao dịch 5 triệu là bình thường với khách hàng A (thường chi 4–6 triệu) nhưng bất thường với khách hàng B (thường chi 200 nghìn). Feature **tương đối so với lịch sử của chính khách hàng** rất mạnh:

$$\text{amount\_ratio} = \frac{\text{amount}}{\text{trung bình amount của khách hàng trong quá khứ}}$$

In [ ]:
n_cust, n_tx = 2000, 60000
cust_level = rng.lognormal(np.log(500), 1.0, n_cust)                      # mức chi tiêu điển hình mỗi khách (nghìn VND)
cust = rng.integers(0, n_cust, n_tx)
t = np.sort(rng.uniform(0, 365, n_tx))
fraud = rng.random(n_tx) < 0.01
amount = cust_level[cust] * rng.lognormal(0, 0.3, n_tx)
amount[fraud] *= rng.uniform(3, 10, fraud.sum())                           # gian lận: lớn bất thường SO VỚI CHÍNH KHÁCH ĐÓ
tx = pd.DataFrame({"customer": cust, "t": t, "amount": amount, "fraud": fraud.astype(int)}).sort_values("t").reset_index(drop=True)

# trung bình chi tiêu QUÁ KHỨ của khách hàng (chỉ dùng các giao dịch trước đó → không rò rỉ)
g = tx.groupby("customer")["amount"]
tx["past_mean"] = (g.cumsum() - tx["amount"]) / g.cumcount().replace(0, np.nan)
tx["past_count"] = g.cumcount()
tx["amount_ratio"] = tx["amount"] / tx["past_mean"]
tx_eval = tx[tx.past_count >= 5]                                           # cần đủ lịch sử
feats = {"chỉ log(amount)": np.log(tx_eval[["amount"]]),
         "log(amount) + log(amount_ratio)": np.c_[np.log(tx_eval.amount), np.log(tx_eval.amount_ratio)]}
for name, Xf in feats.items():
    auc = cross_val_score(HistGradientBoostingClassifier(random_state=0), Xf, tx_eval.fraud, cv=cv5, scoring="roc_auc").mean()
    print(f"{name:32s} ROC-AUC = {auc:.4f}")

> ⚠️ **Rò rỉ khi tổng hợp:** nếu tính trung bình của khách hàng trên **toàn bộ** dữ liệu (kể cả giao dịch tương lai và chính giao dịch đang xét), feature chứa thông tin không có tại thời điểm dự đoán. Luôn tổng hợp **chỉ từ quá khứ** (`cumsum` trừ chính nó, `shift`, rolling window có `closed="left"`), và khi CV theo nhóm/thời gian thì tính lại trong từng fold.

Các aggregation phổ biến: count/sum/mean/std/max theo khách hàng trong 1h/24h/7 ngày, số merchant khác nhau trong 24h, thời gian kể từ giao dịch trước, độ lệch so với trung bình nhóm (z-score trong nhóm).

## 🧠 Kiểm tra nhanh

1. Vì sao Logistic Regression cần feature `young × app × discount` còn HistGB thì không?
2. Mã hoá chu kỳ (sin, cos) của tháng (1–12) là gì? Vì sao cần cả sin và cos?
3. Feature "trung bình chi tiêu của khách hàng" tính trên toàn bộ dữ liệu có vấn đề gì?
4. Kể 3 feature thời gian hữu ích cho dự báo doanh số siêu thị.

## 📌 Tóm tắt

- Model tuyến tính cần feature tương tác/phi tuyến do ta tạo; cây tự học tương tác nhưng vẫn hưởng lợi từ feature miền.
- Thời gian: tách thành phần, mã hoá chu kỳ, khoảng cách tới sự kiện.
- Aggregation theo thực thể rất mạnh — nhưng chỉ tính từ quá khứ để tránh rò rỉ.

## 📚 Tài liệu tham khảo

- Kuhn & Johnson — *Feature Engineering and Selection*, chương 7–9
- Kaggle Learn — Feature Engineering
- Bahnsen et al. (2016) — *Feature engineering strategies for credit card fraud detection*